# Model 8: Ensemble Model (Window=7, Horizon=1)
This notebook demonstrates an ensemble approach for Bitcoin price prediction using a window of 7 days to predict the next day.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# Load Bitcoin historical data
df = pd.read_csv('BTC_USD_2013-10-01_2021-05-18-CoinDesk.csv', parse_dates=['Date'], index_col=['Date'])
bitcoin_prices = pd.DataFrame(df['Closing Price (USD)']).rename(columns={'Closing Price (USD)': 'Price'})
prices = bitcoin_prices['Price'].to_numpy()

In [ ]:
# Utility function for plotting
def plot_time_series(timesteps, values, format='.', start=0, end=None, label=None):
    plt.plot(timesteps[start:end], values[start:end], format, label=label)
    plt.xlabel('Time')
    plt.ylabel('BTC Price')
    if label:
        plt.legend(fontsize=14)
    plt.grid(True)

In [ ]:
# Windowing functions
def get_labelled_windows(x, horizon=1):
    return x[:, :-horizon], x[:, -horizon:]
def make_windows(x, window_size=7, horizon=1):
    window_step = np.expand_dims(np.arange(window_size+horizon), axis=0)
    window_indexes = window_step + np.expand_dims(np.arange(len(x)-(window_size+horizon-1)), axis=0).T
    windowed_array = x[window_indexes]
    windows, labels = get_labelled_windows(windowed_array, horizon=horizon)
    return windows, labels
def make_train_test_splits(windows, labels, test_split=0.2):
    split_size = int(len(windows) * (1-test_split))
    train_windows = windows[:split_size]
    train_labels = labels[:split_size]
    test_windows = windows[split_size:]
    test_labels = labels[split_size:]
    return train_windows, test_windows, train_labels, test_labels

In [ ]:
# Prepare windowed data
HORIZON = 1
WINDOW_SIZE = 7
full_windows, full_labels = make_windows(prices, window_size=WINDOW_SIZE, horizon=HORIZON)
train_windows, test_windows, train_labels, test_labels = make_train_test_splits(full_windows, full_labels)

## Build and Train Ensemble Models

In [ ]:
from tensorflow.keras import layers
tf.random.set_seed(42)
# Dense Model
dense_model = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(WINDOW_SIZE,)),
    layers.Dense(HORIZON)
])
dense_model.compile(loss='mae', optimizer=tf.keras.optimizers.Adam())
dense_model.fit(train_windows, train_labels, epochs=100, verbose=0, batch_size=128, validation_data=(test_windows, test_labels))
# Conv1D Model
conv1d_model = tf.keras.Sequential([
    layers.Reshape((WINDOW_SIZE, 1), input_shape=(WINDOW_SIZE,)),
    layers.Conv1D(128, kernel_size=3, activation='relu'),
    layers.Flatten(),
    layers.Dense(HORIZON)
])
conv1d_model.compile(loss='mae', optimizer=tf.keras.optimizers.Adam())
conv1d_model.fit(train_windows, train_labels, epochs=100, verbose=0, batch_size=128, validation_data=(test_windows, test_labels))
# LSTM Model
lstm_model = tf.keras.Sequential([
    layers.Lambda(lambda x: tf.expand_dims(x, axis=1), input_shape=(WINDOW_SIZE,)),
    layers.LSTM(128, activation='relu'),
    layers.Dense(HORIZON)
])
lstm_model.compile(loss='mae', optimizer=tf.keras.optimizers.Adam())
lstm_model.fit(train_windows, train_labels, epochs=100, verbose=0, batch_size=128, validation_data=(test_windows, test_labels))

## Ensemble Predictions

In [ ]:
def make_preds(model, input_data):
    forecast = model.predict(input_data)
    return tf.squeeze(forecast)
dense_preds = make_preds(dense_model, test_windows)
conv1d_preds = make_preds(conv1d_model, test_windows)
lstm_preds = make_preds(lstm_model, test_windows)
ensemble_preds = (dense_preds + conv1d_preds + lstm_preds) / 3

## Evaluate Ensemble Model

In [ ]:
def mean_absolute_scaled_error(y_true, y_pred):
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    mae_naive_no_season = tf.reduce_mean(tf.abs(y_true[1:] - y_true[:-1]))
    return mae / mae_naive_no_season
def evaluate_preds(y_true, y_pred):
    y_true = tf.cast(y_true, dtype=tf.float32)
    y_pred = tf.cast(y_pred, dtype=tf.float32)
    mae = tf.keras.metrics.mean_absolute_error(y_true, y_pred)
    mse = tf.keras.metrics.mean_squared_error(y_true, y_pred)
    rmse = tf.sqrt(mse)
    mape = tf.keras.metrics.mean_absolute_percentage_error(y_true, y_pred)
    mase = mean_absolute_scaled_error(y_true, y_pred)
    return {"mae": mae.numpy(), "mse": mse.numpy(), "rmse": rmse.numpy(), "mape": mape.numpy(), "mase": mase.numpy()}

In [ ]:
ensemble_results = evaluate_preds(y_true=tf.squeeze(test_labels), y_pred=ensemble_preds)
print(ensemble_results)

## Visualize Ensemble Predictions

In [ ]:
offset = 300
plt.figure(figsize=(10, 7))
plot_time_series(timesteps=bitcoin_prices.index[-len(test_windows):], values=test_labels[:, 0], start=offset, label='Test_data')
plot_time_series(timesteps=bitcoin_prices.index[-len(test_windows):], values=ensemble_preds, start=offset, format='-', label='ensemble_preds')